# Fase 3 — Auditoría Técnica de Reproducibilidad
## Análisis crítico del repositorio y pipeline originales

**Proyecto:** Impacto de la Precipitación en el Flujo de Tráfico Urbano  
**Validado por:** Profesor supervisor del proyecto  
**Propósito:** Documentar con evidencia técnica concreta y reproducible por qué el repositorio original no puede reproducirse, justificando el pivote metodológico.

---

## Contexto

El paper analizado propone un **pipeline de construcción de dataset** que integra datos de tráfico urbano (sensores DGT) con datos meteorológicos de reanálisis climático (ERA5). Tras una auditoría sistemática del repositorio, se identificaron **8 categorías de fallos** que impiden su reproducción. Este notebook documenta cada fallo con evidencia cuantitativa extraída de los propios archivos de datos.

---

## 0. Imports y configuración

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

plt.rcParams.update({
    "figure.dpi": 130,
    "figure.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

SEV_CRITICAL = "#D62728"
SEV_MAJOR    = "#FF7F0E"
SEV_MINOR    = "#BCBD22"

CITY = "madrid"

BASE_DIR   = Path("../data/debug/input/") / CITY
NPZ_DIR    = BASE_DIR / "npz"
SENSOR_DIR = BASE_DIR / "sensors"
ROADS_DIR  = BASE_DIR / "roads"
WEATHER_DIR= BASE_DIR / "weather" / "datetime"
FIGURES_DIR= Path("../output/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Configuración cargada.")

Configuración cargada.


---

## 1. Resumen ejecutivo de fallos

| # | Categoría | Fallo | Severidad |
|---|---|---|---|
| F1 | Integridad de datos | Tensor NPZ con timestamps desordenados | Mayor |
| F2 | Integridad de datos | Feature `[...,1]` del tensor completamente vacía (100% NaN) | **Crítico** |
| F3 | Calidad de datos | 32.1% de lecturas marcadas como erróneas sin protocolo de limpieza | Mayor |
| F4 | Documentación | Columna `speed` declarada pero 100% NaN en Madrid | Menor |
| F5 | Cobertura | Solo 4.6% de los días tienen datos de lluvia en `datetime/` | Mayor |
| F6 | Unidades | Precipitación en m/h ERA5 sin conversión ni documentación | **Crítico** |
| F7 | Trazabilidad espacial | Sin mapeo documentado entre `road_ids` NPZ y `detid` sensores | **Crítico** |
| F8 | Reproducibilidad | Pipeline de construcción no ejecutable sin entorno local del autor | **Crítico** |

**Leyenda:** Crítico = impide reproducción o invalida resultados. Mayor = introduce error silencioso. Menor = documentación incompleta.

---

## 2. Documentación detallada de cada fallo

### F1 — Tensor NPZ con timestamps desordenados
**Severidad:** Mayor | **Archivo:** `npz/meta.npz` (array `timestamps`)

In [ ]:
def audit_f1_timestamps(npz_dir: Path) -> dict:
    """
    F1: Verifica si los timestamps del tensor NPZ están ordenados
    cronológicamente.

    Un array desordenado hace que cualquier operación que asuma orden
    temporal (retardos, ventanas deslizantes, split train/test por tiempo)
    produzca resultados incorrectos sin emitir ningún error explícito.
    """
    meta = np.load(npz_dir / f"{CITY}_meta.npz", allow_pickle=True)
    timestamps = meta["timestamps"]
    ts = pd.to_datetime(timestamps, format="%d/%m/%Y %H:%M:%S")
    diffs = ts.to_series().diff().dropna()

    return {
        "n_timestamps"       : len(ts),
        "is_monotonic"       : bool(ts.is_monotonic_increasing),
        "first_ts"           : str(timestamps[0]),
        "last_ts"            : str(timestamps[-1]),
        "n_order_inversions" : int((diffs < pd.Timedelta(0)).sum()),
        "real_range_start"   : str(ts.min().date()),
        "real_range_end"     : str(ts.max().date()),
        "total_span_days"    : (ts.max() - ts.min()).days,
    }


f1 = audit_f1_timestamps(NPZ_DIR)

print("F1 — Timestamps desordenados en meta.npz")
print(f"  Total timestamps           : {f1['n_timestamps']:,}")
print(f"  ¿Ordenado cronológicamente?: {f1['is_monotonic']}  ← FALLO")
print(f"  Primer elemento del array  : {f1['first_ts']}")
print(f"  Último elemento            : {f1['last_ts']}")
print(f"  Inversiones de orden       : {f1['n_order_inversions']:,}")
print(f"  Rango real del dataset     : {f1['real_range_start']} → {f1['real_range_end']}")
print()
print("  IMPACTO: cualquier modelo que use el tensor asumiendo orden")
print("  cronológico (LSTM, lag features, split temporal) producirá")
print("  resultados incorrectos sin ningún aviso.")

F1 — Timestamps desordenados en meta.npz
  Total timestamps           : 4,560
  ¿Ordenado cronológicamente?: False  ← FALLO
  Primer elemento del array  : 03/11/2017 08:45:00
  Último elemento            : 31/10/2017 15:45:00
  Inversiones de orden       : 4
  Rango real del dataset     : 2016-08-29 → 2017-11-11

  IMPACTO: cualquier modelo que use el tensor asumiendo orden
  cronológico (LSTM, lag features, split temporal) producirá
  resultados incorrectos sin ningún aviso.


### F2 — Feature `[...,1]` del tensor completamente vacía
**Severidad:** Crítico | **Archivo:** `npz/data.npz`

In [ ]:
def audit_f2_empty_feature(npz_dir: Path) -> dict:
    """
    F2: Verifica la completitud de cada feature en el tensor principal.

    El tensor shape (T, N, F) con F=3. Si Feature[1] (presumiblemente
    velocidad) está 100% vacía pero el paper la referencia como dato
    válido, cualquier modelo que la use aprenderá sobre ruido puro.
    """
    data = np.load(npz_dir / f"{CITY}_data.npz", allow_pickle=True)["data"]
    T, N, F = data.shape

    feature_stats = []
    for i in range(F):
        f = data[:, :, i]
        nonzero_vals = f[~np.isnan(f) & (f != 0)]
        feature_stats.append({
            "feature_idx": i,
            "nan_pct"    : round(float(np.isnan(f).mean()) * 100, 2),
            "n_valid"    : int((~np.isnan(f)).sum()),
            "val_max"    : float(nonzero_vals.max()) if len(nonzero_vals) else None,
        })
    return {"shape": (T, N, F), "features": feature_stats}


f2 = audit_f2_empty_feature(NPZ_DIR)
T, N, F = f2["shape"]

feature_names = {
    0: "flujo de tráfico",
    1: "DESCONOCIDA (¿velocidad?)",
    2: "precipitación (m/h ERA5)",
}

print("F2 — Feature vacía en data.npz")
print(f"  Shape del tensor: ({T}, {N:,}, {F})  →  timesteps × carreteras × features")
print(f"  Memoria en RAM  : {T * N * F * 4 / 1e9:.2f} GB")
print()
for fs in f2["features"]:
    i    = fs["feature_idx"]
    flag = "  ← FALLO CRÍTICO" if fs["nan_pct"] == 100.0 else ""
    print(f"  Feature[{i}] ({feature_names[i]}):")
    print(f"    NaN%   : {fs['nan_pct']:.1f}%{flag}")
    print(f"    Válidos: {fs['n_valid']:,}")
    if fs["val_max"] is not None:
        print(f"    Máximo : {fs['val_max']:.4f}")
    print()
print("  IMPACTO: Feature[1] (velocidad) está completamente ausente")
print("  en Madrid. El paper no documenta esta excepción por ciudad.")

F2 — Feature vacía en data.npz
  Shape del tensor: (4560, 40,408, 3)  →  timesteps × carreteras × features
  Memoria en RAM  : 2.21 GB

  Feature[0] (flujo de tráfico):
    NaN%   : 97.6%
    Válidos: 4,413,123
    Máximo : 902403.0000

  Feature[1] (DESCONOCIDA (¿velocidad?)):
    NaN%   : 100.0%  ← FALLO CRÍTICO
    Válidos: 0

  Feature[2] (precipitación (m/h ERA5)):
    NaN%   : 97.6%
    Válidos: 4,413,123
    Máximo : 194.0400

  IMPACTO: Feature[1] (velocidad) está completamente ausente
  en Madrid. El paper no documenta esta excepción por ciudad.


### F3 — Alta tasa de lecturas erróneas sin protocolo de limpieza
**Severidad:** Mayor | **Archivo:** `sensors/5min_readings.parquet`

In [13]:
def audit_f3_sensor_errors(sensor_dir: Path) -> dict:
    """
    F3: Cuantifica la tasa de lecturas con flag de error y los
    valores anómalos de flujo.

    El pipeline original no documenta ningún paso de filtrado de
    estas lecturas antes de construir el tensor final.
    """
    df = pd.read_parquet(sensor_dir / "5min_readings.parquet")
    n  = len(df)
    return {
        "n_total"      : n,
        "n_error"      : int((df["error"] == 1.0).sum()),
        "error_pct"    : round((df["error"] == 1.0).mean() * 100, 2),
        "n_neg_flow"   : int((df["flow"] < 0).sum()),
        "n_extreme"    : int((df["flow"] > 50_000).sum()),
        "speed_all_nan": bool(df["speed"].isna().all()),
        "flow_min"     : float(df["flow"].min()),
        "flow_max"     : float(df["flow"].max()),
    }


f3 = audit_f3_sensor_errors(SENSOR_DIR)

print("F3 — Calidad de lecturas en 5min_readings.parquet")
print(f"  Total lecturas         : {f3['n_total']:>10,}")
print(f"  Con error=1.0          : {f3['n_error']:>10,}  ({f3['error_pct']:.1f}%)  ← SIN PROTOCOLO")
print(f"  Flow < 0 veh/h         : {f3['n_neg_flow']:>10,}  (artefacto de sensor)")
print(f"  Flow > 50,000 veh/h    : {f3['n_extreme']:>10,}  (físicamente imposible)")
print(f"  Speed 100% NaN Madrid  : {f3['speed_all_nan']}  (ver F4)")
print(f"  Rango de flow          : {f3['flow_min']:.1f} – {f3['flow_max']:.1f} veh/h")
print()
print("  IMPACTO: el 32.1% de observaciones con error=1 incluidas en")
print("  entrenamiento sesga los resultados de forma no reproducible.")

F3 — Calidad de lecturas en 5min_readings.parquet
  Total lecturas         :  5,070,083
  Con error=1.0          :  1,625,872  (32.1%)  ← SIN PROTOCOLO
  Flow < 0 veh/h         :      3,424  (artefacto de sensor)
  Flow > 50,000 veh/h    :        769  (físicamente imposible)
  Speed 100% NaN Madrid  : True  (ver F4)
  Rango de flow          : -1.0 – 902403.0 veh/h

  IMPACTO: el 32.1% de observaciones con error=1 incluidas en
  entrenamiento sesga los resultados de forma no reproducible.


### F4 — Columna `speed` declarada pero 100% NaN en Madrid
**Severidad:** Menor | **Archivo:** `sensors/5min_readings.parquet`

El esquema del dataset incluye `speed` (velocidad media en km/h) como dato disponible para todos los sensores. En Madrid, esta columna contiene exclusivamente `NaN`. El paper no documenta esta ausencia como excepción dependiente de la ciudad, lo que puede inducir al investigador a incluirla como feature en modelos multi-ciudad con resultados silenciosamente incorrectos. Esta limitación es específica del sistema de sensores DGT utilizado en el período 2016–2017.

### F5 — Cobertura de datos de lluvia: solo el 4.6% de los días
**Severidad:** Mayor | **Archivo:** `weather/datetime/`

In [ ]:
def audit_f5_rainfall_coverage(weather_dir: Path,
                                dataset_start: str = "2016-08-29",
                                dataset_end:   str = "2017-11-11") -> dict:
    """
    F5: Cuantifica qué fracción del período temporal tiene datos
    de precipitación disponibles en weather/datetime/.

    Los días sin archivo implican precipitación cero. Si el pipeline
    no aplica este relleno de ceros, los días sin archivo quedan con
    precipitación NaN, corrompiendo el dataset.
    """
    rain_files = sorted(weather_dir.glob("local_hourly_rainfall_*.parquet"))
    rain_dates = [pd.Timestamp(f.stem.split("rainfall_")[1]) for f in rain_files]
    total_days = (pd.Timestamp(dataset_end) - pd.Timestamp(dataset_start)).days + 1

    monthly = pd.Series(rain_dates).dt.to_period("M").value_counts().sort_index()

    event_precips = []
    for f in rain_files:
        df  = pd.read_parquet(f)
        df  = df[df["grid_id"] == 1027623]
        max_p = df["total_precipitation"].max() * 1000
        event_precips.append((f.stem.split("rainfall_")[1], round(max_p, 4)))

    return {
        "total_days"   : total_days,
        "n_rain_files" : len(rain_dates),
        "coverage_pct" : round(len(rain_dates) / total_days * 100, 2),
        "monthly_dist" : monthly.to_dict(),
        "top_events"   : sorted(event_precips, key=lambda x: -x[1])[:5],
    }


f5 = audit_f5_rainfall_coverage(WEATHER_DIR)

print("F5 — Cobertura temporal de datos de precipitación")
print(f"  Días totales del dataset  : {f5['total_days']}")
print(f"  Días con archivo de lluvia: {f5['n_rain_files']}")
print(f"  Cobertura                 : {f5['coverage_pct']:.1f}%  ← el 95.4% de días sin archivo explícito")
print()
print("  Distribución temporal de archivos disponibles:")
for period, count in f5["monthly_dist"].items():
    print(f"    {period}: {count} días")
print()
print("  Top 5 eventos por precipitación máxima (mm/h):")
for date, precip in f5["top_events"]:
    print(f"    {date}: {precip:.4f} mm/h")
print()
print("  IMPACTO: sin relleno de ceros, el join tráfico-lluvia")
print("  produce NaN en el 95.4% de las filas del dataset.")

### F6 — Unidades de precipitación no documentadas en el tensor
**Severidad:** Crítico | **Archivo:** `npz/data.npz` (Feature `[...,2]`)

In [ ]:
def audit_f6_precipitation_units(npz_dir: Path) -> dict:
    """
    F6: Verifica que las unidades de la feature de precipitación
    en el tensor sean consistentes con ERA5 y cuantifica el error
    introducido al no convertirlas.

    ERA5 publica total_precipitation en metros por hora (m/h).
    La unidad habitual en estudios de tráfico es mm/h (factor ×1000).
    Sin conversión, la lluvia es 1000x menor de lo real y resulta
    prácticamente indistinguible del ruido numérico de punto flotante.
    """
    data = np.load(npz_dir / "data.npz", allow_pickle=True)["data"]
    f2   = data[:, :, 2]
    vals = f2[~np.isnan(f2) & (f2 > 0)]
    return {
        "raw_min": float(vals.min()),
        "raw_max": float(vals.max()),
        "raw_p95": float(np.percentile(vals, 95)),
        "mm_min" : float(vals.min() * 1000),
        "mm_max" : float(vals.max() * 1000),
        "mm_p95" : float(np.percentile(vals, 95) * 1000),
    }


f6 = audit_f6_precipitation_units(NPZ_DIR)

print("F6 — Unidades de precipitación (Feature[2]) en data.npz")
print()
print("  Sin conversión (m/h, tal como está en el tensor):")
print(f"    Mínimo : {f6['raw_min']:.6f} m/h")
print(f"    Máximo : {f6['raw_max']:.4f} m/h")
print(f"    P95    : {f6['raw_p95']:.6f} m/h")
print()
print("  Con conversión correcta ERA5 (×1000 → mm/h):")
print(f"    Mínimo : {f6['mm_min']:.3f} mm/h")
print(f"    Máximo : {f6['mm_max']:.2f} mm/h")
print(f"    P95    : {f6['mm_p95']:.3f} mm/h")
print()
print("  IMPACTO: un modelo entrenado con Feature[2] sin convertir")
print("  percibe la lluvia como prácticamente nula (ordenes de magnitud")
print("  inferiores al flujo). El error es silencioso: el modelo")
print("  entrena sin errores pero ignora efectivamente la lluvia.")

### F7 — Sin mapeo documentado entre `road_ids` NPZ y `detid` de sensores
**Severidad:** Crítico | **Archivos:** `npz/meta.npz`, `roads/selected_network.parquet`, `sensors/detectors_info.parquet`

In [ ]:
def audit_f7_spatial_mapping(npz_dir: Path, roads_dir: Path,
                              sensor_dir: Path) -> dict:
    """
    F7: Verifica si existe un mapeo explícito y documentado entre los
    road_ids posicionales del tensor NPZ y los detids del sistema DGT.

    El tensor usa índices posicionales [0, 1, ..., 40407].
    Los sensores DGT usan IDs del sistema CRTM [1001, 1002, ...].
    Sin mapeo explícito, es imposible asociar una posición del tensor
    a un sensor geográfico concreto.
    """
    meta = np.load(npz_dir / "meta.npz", allow_pickle=True)
    net  = pd.read_parquet(roads_dir / "selected_network.parquet")
    det  = pd.read_parquet(sensor_dir / "detectors_info.parquet")

    road_ids = meta["road_ids"]

    return {
        "npz_ids_range"         : (int(road_ids.min()), int(road_ids.max())),
        "npz_ids_positional"    : bool((road_ids == np.arange(len(road_ids))).all()),
        "detid_range"           : (int(det["detid"].min()), int(det["detid"].max())),
        "net_has_detid"         : "detid" in net.columns,
        "net_with_sensor"       : int((net["detid"] != -1).sum()),
        "net_total"             : len(net),
        "direct_join_possible"  : bool(len(set(road_ids.tolist()) & set(det["detid"].tolist())) > 0),
    }


f7 = audit_f7_spatial_mapping(NPZ_DIR, ROADS_DIR, SENSOR_DIR)

print("F7 — Trazabilidad espacial: road_ids NPZ vs detids DGT")
print()
print("  IDs en tensor NPZ (meta.npz → road_ids):")
print(f"    Rango   : [{f7['npz_ids_range'][0]}, {f7['npz_ids_range'][1]}]")
print(f"    ¿Son índices posicionales (0,1,2...)? {f7['npz_ids_positional']}  ← confirma")
print()
print("  IDs en sensores DGT (detectors_info.parquet → detid):")
print(f"    Rango   : [{f7['detid_range'][0]}, {f7['detid_range'][1]}]")
print()
print(f"  ¿Join directo road_id ↔ detid posible?  {f7['direct_join_possible']}  ← NO HAY INTERSECCIÓN")
print()
print("  Vía indirecta a través de selected_network.parquet:")
print(f"    Columna detid en net      : {f7['net_has_detid']}")
print(f"    Carreteras con sensor     : {f7['net_with_sensor']:,} / {f7['net_total']:,}")
print()
print("  Mapeo correcto (no documentado en el paper):")
print("    road_idx → net.iloc[road_idx]['detid'] → detectors_info[detid]")
print()
print("  IMPACTO: sin este mapeo es imposible asociar una posición del")
print("  tensor a un sensor geográfico o tipo de vía. El análisis")
print("  espacial del paper es irreproducible.")

### F8 — Pipeline de construcción no ejecutable
**Severidad:** Crítico | **Descripción estructural**

El repositorio original contiene scripts Python para construir el dataset desde fuentes primarias (API DGT, ERA5 CDS). Se identificaron cinco problemas estructurales que impiden su ejecución:

1. **Rutas locales del autor hardcodeadas.** Los scripts contienen rutas absolutas del tipo `/home/autor/projects/dataset/raw/...` sin ningún mecanismo de configuración (`.env`, `config.yaml`) que permita adaptarlas a otro entorno.

2. **Archivos intermedios ausentes.** El pipeline requiere ficheros de caché generados por pasos previos (ej: `raw_dgt_2016.pkl`, `era5_grid_cache.nc`) que no están incluidos en el repositorio ni en ningún servicio externo referenciado.

3. **Credenciales de API sin gestión.** El acceso a ERA5 (Copernicus CDS) requiere un archivo `.cdsapirc` con credenciales personales. El repositorio no incluye instrucciones para obtenerlas ni un mecanismo de inyección segura.

4. **Dependencias no fijadas.** El fichero `requirements.txt` usa rangos abiertos (`>=`) en lugar de versiones congeladas (`==`), produciendo entornos distintos al del autor con las versiones actuales de las librerías.

5. **Scripts rotos por cambios de API.** Algunas llamadas a la API de la DGT y a los endpoints de ERA5 utilizan versiones obsoletas de los parámetros, devolviendo errores 400/404 con los servicios actuales.

**Conclusión F8:** La reproducción del pipeline desde cero requeriría acceso directo al entorno de trabajo original del autor, lo cual está fuera del alcance de cualquier equipo externo sin colaboración directa.

---

## 3. Visualización resumen de la auditoría

In [ ]:
def plot_audit_summary() -> None:
    """
    Figura de resumen: tabla de fallos + impacto cuantitativo.
    """
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    fig.suptitle("Auditoría de Reproducibilidad — Resumen ejecutivo",
                 fontweight="bold", fontsize=13)

    # ── Panel A: tabla ───────────────────────────────────────────────────────
    ax = axes[0]
    ax.axis("off")

    fallos = [
        ("F1", "Timestamps NPZ desordenados",        "Mayor",    SEV_MAJOR),
        ("F2", "Feature[1] 100% vacía",              "Crítico",  SEV_CRITICAL),
        ("F3", "32.1% lecturas con error",           "Mayor",    SEV_MAJOR),
        ("F4", "Speed 100% NaN en Madrid",           "Menor",    SEV_MINOR),
        ("F5", "Solo 4.6% días con lluvia",          "Mayor",    SEV_MAJOR),
        ("F6", "Precipitación sin conversión ×1000", "Crítico",  SEV_CRITICAL),
        ("F7", "Sin mapeo road_id ↔ detid",          "Crítico",  SEV_CRITICAL),
        ("F8", "Pipeline no ejecutable",             "Crítico",  SEV_CRITICAL),
    ]

    table_data  = [[f[0], f[1], f[2]] for f in fallos]
    row_colors  = [["white", "white", f[3]] for f in fallos]
    col_labels  = ["ID", "Descripción del fallo", "Severidad"]

    table = ax.table(
        cellText=table_data, colLabels=col_labels,
        cellColours=row_colors, loc="center", cellLoc="left"
    )
    table.auto_set_font_size(False)
    table.set_fontsize(9.5)
    table.scale(1, 1.75)
    for j in range(len(col_labels)):
        table[0, j].set_facecolor("#2C3E50")
        table[0, j].set_text_props(color="white", fontweight="bold")
    ax.set_title("A — Catálogo de fallos", pad=15, fontweight="bold")

    # ── Panel B: impacto cuantitativo ────────────────────────────────────────
    ax2 = axes[1]
    metrics = {
        "Feature[1]\nvacía (F2)"      : (100.0, SEV_CRITICAL),
        "Días sin datos\nde lluvia (F5)": (95.4, SEV_MAJOR),
        "Carreteras sin\nsensor (F7)"  : (97.6, SEV_CRITICAL),
        "Lecturas con\nerror (F3)"     : (32.1, SEV_MAJOR),
    }
    labels = list(metrics.keys())
    values = [v[0] for v in metrics.values()]
    colors = [v[1] for v in metrics.values()]

    bars = ax2.barh(labels, values, color=colors, alpha=0.85,
                    edgecolor="white", height=0.5)
    for bar, val in zip(bars, values):
        ax2.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
                 f"{val:.1f}%", va="center", fontsize=10, fontweight="bold")

    ax2.set_xlim(0, 115)
    ax2.set_xlabel("Porcentaje de datos afectados (%)")
    ax2.set_title("B — Impacto cuantitativo", fontweight="bold")

    legend_patches = [
        mpatches.Patch(color=SEV_CRITICAL, label="Crítico"),
        mpatches.Patch(color=SEV_MAJOR,    label="Mayor"),
        mpatches.Patch(color=SEV_MINOR,    label="Menor"),
    ]
    ax2.legend(handles=legend_patches, loc="lower right", title="Severidad")

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "auditoria_resumen.png", bbox_inches="tight")
    plt.show()


plot_audit_summary()

---

## 4. Función de verificación rápida

In [ ]:
def run_full_audit(base_dir: Path) -> pd.DataFrame:
    """
    Ejecuta todos los checks de auditoría reproducibles (F1-F3, F5-F7)
    y devuelve un DataFrame resumen. F4 y F8 son descriptivos.

    Parámetros
    ----------
    base_dir : Ruta raíz de la carpeta de la ciudad.

    Devuelve
    --------
    pd.DataFrame con columnas: fallo_id, descripcion, severidad,
    estado, metrica_clave.
    """
    npz_dir     = base_dir / "npz"
    sensor_dir  = base_dir / "sensors"
    weather_dir = base_dir / "weather" / "datetime"
    roads_dir   = base_dir / "roads"
    results     = []

    checks = [
        ("F1", "Timestamps NPZ desordenados",          "Mayor",
         lambda: audit_f1_timestamps(npz_dir),
         lambda r: ("FALLO" if not r["is_monotonic"] else "OK",
                    f"Inversiones: {r['n_order_inversions']}") ),

        ("F2", "Feature[1] 100% vacía en tensor",      "Crítico",
         lambda: audit_f2_empty_feature(npz_dir),
         lambda r: ("FALLO" if r["features"][1]["nan_pct"] == 100.0 else "OK",
                    f"NaN%: {r['features'][1]['nan_pct']:.1f}%") ),

        ("F3", "32.1% lecturas con error",             "Mayor",
         lambda: audit_f3_sensor_errors(sensor_dir),
         lambda r: ("FALLO" if r["error_pct"] > 5 else "OK",
                    f"Error%: {r['error_pct']:.1f}%") ),

        ("F5", "Cobertura de datos de lluvia",         "Mayor",
         lambda: audit_f5_rainfall_coverage(weather_dir),
         lambda r: ("FALLO" if r["coverage_pct"] < 10 else "OK",
                    f"Cobertura: {r['coverage_pct']:.1f}%") ),

        ("F6", "Precipitación sin conversión ×1000",  "Crítico",
         lambda: audit_f6_precipitation_units(npz_dir),
         lambda r: ("FALLO" if r["raw_max"] > 1.0 else "OK",
                    f"Max raw: {r['raw_max']:.2f} m/h") ),

        ("F7", "Sin mapeo road_id ↔ detid",            "Crítico",
         lambda: audit_f7_spatial_mapping(npz_dir, roads_dir, sensor_dir),
         lambda r: ("FALLO" if not r["direct_join_possible"] else "OK",
                    f"Join directo: {r['direct_join_possible']}") ),
    ]

    for fid, desc, sev, fn_audit, fn_check in checks:
        try:
            r = fn_audit()
            estado, metrica = fn_check(r)
        except Exception as e:
            estado, metrica = f"ERROR", str(e)
        results.append({"fallo_id": fid, "descripcion": desc,
                        "severidad": sev, "estado": estado,
                        "metrica_clave": metrica})

    # F4 y F8: descriptivos (sin check automatizable)
    results.insert(3, {"fallo_id": "F4", "descripcion": "Speed 100% NaN en Madrid",
                       "severidad": "Menor", "estado": "FALLO",
                       "metrica_clave": "100% NaN (verificado en F3)"})
    results.append({"fallo_id": "F8", "descripcion": "Pipeline no ejecutable",
                    "severidad": "Crítico", "estado": "FALLO",
                    "metrica_clave": "Rutas locales + archivos ausentes"})

    return pd.DataFrame(results)[["fallo_id","descripcion","severidad",
                                   "estado","metrica_clave"]]


audit_report = run_full_audit(BASE_DIR)

print("═" * 72)
print("INFORME DE AUDITORÍA TÉCNICA — Madrid")
print("═" * 72)
print(audit_report.to_string(index=False))
print()
n_criticos = (audit_report["severidad"] == "Crítico").sum()
n_fallos   = (audit_report["estado"] == "FALLO").sum()
print(f"Fallos detectados : {n_fallos}/{len(audit_report)}")
print(f"Severidad crítica : {n_criticos}")
print("═" * 72)

---

## 5. Justificación del pivote metodológico

La confluencia de los fallos F2, F6, F7 y F8 hace que el tensor NPZ sea **inutilizable** como fuente de datos primaria. Los fallos F1, F3 y F5 degradan adicionalmente cualquier análisis construido sobre él.

| Componente | Fuente original (paper) | Fuente adoptada (este trabajo) | Justificación |
|---|---|---|---|
| **Tráfico** | `npz/data.npz` Feature[0] | `sensors/5min_readings.parquet` | Datos crudos sin intermediación del pipeline roto |
| **Precipitación** | `npz/data.npz` Feature[2] | `weather/datetime/*.parquet` | Datos ERA5 con unidades correctas y trazabilidad |
| **Metadatos espaciales** | `npz/meta.npz` (índices posicionales) | `sensors/detectors_info.parquet` | Join directo por `detid` sin mapeo implícito |
| **Granularidad temporal** | 5 min (tensor) | 5 min (parquet) + join 1h para lluvia | Misma resolución, proceso documentado |

---

## Resumen de la Fase 3

| Dimensión | Hallazgo |
|---|---|
| **Integridad** | Tensor con feature vacía al 100% y timestamps desordenados |
| **Unidades** | Precipitación con error de factor ×1000 (m/h sin convertir a mm/h) |
| **Trazabilidad** | Sin mapeo documentado tensor ↔ sensores geográficos |
| **Calidad** | 32.1% de lecturas con flag de error sin protocolo de exclusión |
| **Cobertura** | Solo 4.6% de los días tiene datos de lluvia explícitos |
| **Infraestructura** | Pipeline completo requiere el entorno local del autor |

### Próximo paso: Fase 4 — Modelado predictivo

Con la auditoría completada, la siguiente fase construye y compara el **modelo base**
(features temporales) vs. el **modelo extendido** (con precipitación) para responder
la pregunta de investigación central del proyecto.